# 06 — Large Language Model (LLM) Scoring of GDELT Headlines
**GeoSentinel Terminal (VARTA) · Team 7 Lambda · SP2026**

Scores Global Database of Events, Language, and Tone (GDELT) headlines for geopolitical
supply chain risk using two local Ollama models:

- **Phi-3 Mini (Microsoft Research):** Fast pass — all headlines, rapid classification
- **Qwen 2.5:14b (Alibaba Cloud):** Deep pass — top 500 highest-tone headlines only

⚠️  **Rule:** Qwen 2.5:14b is PRE-COMPUTED here and cached — never call it on user interaction in the Streamlit app.

Outputs: `data/processed/gdelt.parquet` (with `llm_score` column filled)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

In [ ]:
import json
import requests
import polars as pl
from config import DATA_PROC, OLLAMA_HOST, OLLAMA_DEEP_MODEL, OLLAMA_FAST_MODEL, LLM_MAX_TOKENS
from src.utils import log, save_parquet

# Verify Ollama is running
try:
    resp = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=5)
    models = [m["name"] for m in resp.json().get("models", [])]
    log.info(f"Ollama running — available models: {models}")
except Exception as e:
    raise RuntimeError(f"Ollama not reachable at {OLLAMA_HOST}: {e}\nStart Ollama: open Ollama.app")

for model in [OLLAMA_FAST_MODEL, OLLAMA_DEEP_MODEL]:
    if not any(model in m for m in models):
        log.warning(f"Model '{model}' not found — run: ollama pull {model}")

In [ ]:
# ── Output schema ─────────────────────────────────────────────────────────────
OUTPUT_SCHEMA = {
    "crisis_score": "float 0.0-1.0: probability this headline signals a geopolitical supply chain crisis",
    "sentiment":    "string: negative | neutral | positive",
    "key_entities": "list of strings: countries, assets, or events mentioned",
    "reasoning":    "string: one sentence justification",
}

def score_headline(headline: str, use_deep: bool = False) -> dict:
    """Score one headline using Gemma 4 4B (fast) or Gemma 4 26B (deep)."""
    model = OLLAMA_DEEP_MODEL if use_deep else OLLAMA_FAST_MODEL
    prompt = (
        f"You are a geopolitical supply chain risk analyst. "
        f"Score this headline for geopolitical supply chain crisis risk, considering: "
        f"trade wars, sanctions, critical mineral supply disruptions, semiconductor supply chain, "
        f"energy supply shocks, shipping route disruptions, and military conflicts. "
        f"Return ONLY valid JavaScript Object Notation (JSON) matching: {json.dumps(OUTPUT_SCHEMA)}.\n\n"
        f"Headline: {headline}"
    )
    payload = {"model": model, "prompt": prompt, "stream": False,
               "options": {"num_predict": LLM_MAX_TOKENS}}
    try:
        resp = requests.post(f"{OLLAMA_HOST}/api/generate", json=payload, timeout=60)
        resp.raise_for_status()
        raw = resp.json().get("response", "{}").strip().removeprefix("```json").removesuffix("```").strip()
        result = json.loads(raw)
        return {"crisis_score": float(result.get("crisis_score", 0.5)),
                "sentiment":    str(result.get("sentiment", "neutral")),
                "key_entities": result.get("key_entities", []),
                "reasoning":    str(result.get("reasoning", ""))}
    except Exception as e:
        log.warning(f"LLM error ({model}): {e}")
        return {"crisis_score": 0.5, "sentiment": "neutral", "key_entities": [], "reasoning": "error"}


In [ ]:
# ── Load GDELT headlines ───────────────────────────────────────────────────────
gdelt = pl.read_parquet(DATA_PROC / "gdelt.parquet")
headlines = gdelt["headline"].to_list()
log.info(f"Total headlines to score: {len(headlines):,}")

In [ ]:
# ── Fast pass: Phi-3 Mini on all headlines ────────────────────────────────────
log.info(f"Starting fast pass with Phi-3 Mini ({OLLAMA_FAST_MODEL})...")
fast_scores = []
for i, h in enumerate(headlines):
    result = score_headline(h, use_deep=False)
    fast_scores.append(result["crisis_score"])
    if (i + 1) % 100 == 0:
        log.info(f"  Fast pass: {i+1}/{len(headlines)} headlines scored")

log.info(f"Fast pass complete — mean crisis score: {sum(fast_scores)/len(fast_scores):.3f}")

In [ ]:
# ── Deep pass: Gemma 4 26B on top 500 highest crisis-risk headlines ──────────
# Rank by fast_scores (not GDELT tone — tone column is null for fetched articles).
# Deep model re-scores the 500 headlines the fast pass flagged as highest risk.
top_500_indices = sorted(range(len(fast_scores)), key=lambda i: fast_scores[i], reverse=True)[:500]

deep_scores = fast_scores.copy()  # start from fast scores
log.info(f"Starting deep pass with Gemma 4 26B ({OLLAMA_DEEP_MODEL}) on top 500 headlines...")
for rank, idx in enumerate(top_500_indices):
    result = score_headline(headlines[idx], use_deep=True)
    deep_scores[idx] = result["crisis_score"]  # override fast score with deep score
    if (rank + 1) % 50 == 0:
        log.info(f"  Deep pass: {rank+1}/500")

log.info("Deep pass complete")


In [ ]:
# ── Save scored GDELT ─────────────────────────────────────────────────────────
gdelt_scored = gdelt.with_columns(pl.Series("llm_score", deep_scores))

print(f"Crisis score distribution:")
print(gdelt_scored.select("llm_score").describe())

save_parquet(gdelt_scored, DATA_PROC / "gdelt.parquet", "GDELT with LLM scores")
print("Saved → data/processed/gdelt.parquet (llm_score column filled)")